## Indicator Calculator
### - Moving Averages(MA), Standard Diviation(STD),  Z-Score, Correlation(CORR) and Volatility(VOLA)
### - Delta, Cumulative Delta

### 1 - Import Libraries

In [1]:
import pandas as pd

### 2 - Import Data from Bitcoin Futures csv file and check the dataframe

In [2]:
df = pd.read_csv('bitcoin_futures_raw_data/futures_raw_data.csv')

In [3]:
df.tail()

,datetime,open_price,high_price,low_price,close_price,volume,buy_volume,transactions,buy_transactions,long_short_ratio,...,low_predicted_funding_rate,close_predicted_funding_rate,open_funding_rate,high_funding_rate,low_funding_rate,close_funding_rate,open_open_interest,high_open_interest,low_open_interest,close_open_interest
2206,1758931200,109588.5,109700.0,109021.9,109577.3,35808.071,17875.676,342032,168645.0,1.7917,...,-0.000577,0.003332,-0.000010,0.007821,-0.000010,0.007821,84919.760,85127.114,84517.446,84550.888
2207,1759017600,109577.2,112300.0,109136.5,112119.6,77220.071,39810.148,614121,314616.0,1.7824,...,0.001616,0.002264,0.003314,0.005261,0.003314,0.005261,84550.926,86224.886,84418.636,86090.117
2208,1759104000,112119.7,114377.2,111501.0,114257.1,125365.977,63032.434,983709,487959.0,1.4900,...,-0.001262,0.002353,0.002312,0.002312,0.000032,0.000032,86089.696,88987.440,85641.969,88865.851
2209,1759190400,114257.1,114800.0,112615.3,113988.8,119261.073,60303.437,1063050,529369.0,1.0227,...,0.001904,0.003410,0.002353,0.005893,0.002353,0.002584,88864.528,90374.025,87585.658,88524.384
2210,1759276800,113988.7,114669.0,113899.4,114450.1,19162.092,9376.931,154609,74069.0,1.0683,...,0.001193,0.003825,0.003355,0.003355,0.003355,0.003355,88524.384,89769.610,88326.802,89763.968


In [4]:
df.columns

Index(['datetime', 'open_price', 'high_price', 'low_price', 'close_price',
       'volume', 'buy_volume', 'transactions', 'buy_transactions',
       'long_short_ratio', 'long_long_short_ratio', 's_long_short_ratio',
       'long_liquidation', 'short_liquidation', 'open_predicted_funding_rate',
       'high_predicted_funding_rate', 'low_predicted_funding_rate',
       'close_predicted_funding_rate', 'open_funding_rate',
       'high_funding_rate', 'low_funding_rate', 'close_funding_rate',
       'open_open_interest', 'high_open_interest', 'low_open_interest',
       'close_open_interest'],
      dtype='object')

### 3 - List and Rename Columns to Calculate

In [5]:
# create some new columns
df['volume_delta'] = df.buy_volume - (df.volume - df.buy_volume)
df['cumulative_volume_delta'] = df.volume_delta.cumsum()
df['transaction_delta'] = df.buy_transactions - (df.transactions - df.buy_transactions)
df['cumulative_transaction_delta'] = df.transaction_delta.cumsum()
df['liquidation_delta'] = df.long_liquidation - df.short_liquidation
df['cumulative_liquidation_delta'] = df.liquidation_delta.cumsum()

# create a list of data to calculate the indicators
to_calculate_col = [
    'datetime',
    'close_price',
    'volume',
    'volume_delta',
    'cumulative_volume_delta',
    'transactions',
    'transaction_delta',
    'cumulative_transaction_delta',
    'long_short_ratio',
    'short_liquidation',
    'long_liquidation',
    'liquidation_delta',
    'cumulative_liquidation_delta',
    'close_predicted_funding_rate',
    'close_funding_rate',
    'close_open_interest'
]

### 4 - Function to Calculate the Indicators

In [6]:
# create the function
def indicator_func(df, to_calculate, timeframe, weighter=False, correlator=False):

    # if to weight is not needed
    if not weighter:
        ma =  df[to_calculate].rolling(window=timeframe).mean() # moving average
        std = df[to_calculate].rolling(window=timeframe).std() # standad deviation

    # if needs to me weighted by something
    else:
        ma = (
            (df[to_calculate] * df[weighter]).rolling(window=timeframe).sum() / # moving average
            df[weighter].rolling(window=timeframe).sum()
        )
        std = (
            (((df[to_calculate]** 2) * df[weighter]).rolling(window=timeframe).sum() / # standad deviation
             df[weighter].rolling(window=timeframe).sum()) - (ma ** 2)) ** 0.5
        
    z_score = (df[to_calculate] - ma) / std # z-score
    
    # create a new dataframe with just the datetime    
    df_new = df[['datetime']].copy()

    # add the indicators to new columns     
    df_new[f'{to_calculate}_{timeframe}_ma'] = ma
    df_new[f'{to_calculate}_{timeframe}_std'] = std
    df_new[f'{to_calculate}_{timeframe}_z_score'] = z_score

    # add correlation if needed
    if correlator:
        correlation = df[to_calculate].rolling(window=timeframe).corr(df.correlator) # correlation
        df_new[f'{to_calculate}_{correlator}_{timeframe}_correlation'] = correlation    
    
    # return the result
    return df_new

### 5 - Loop to calculate the Indicators

In [7]:
# list of timeframes
timeframes = [7, 31, 127, 367]

# new dataframe to store the indicators
df_indicators = df[[
    'datetime',
    'volume_delta',
    'cumulative_volume_delta',
    'transaction_delta',
    'cumulative_transaction_delta',
    'liquidation_delta',
    'cumulative_liquidation_delta'    
]]

# loop to call the function for each indicator and each timeframe
for col in to_calculate_col:
    if col != 'datetime':
        for timeframe in timeframes:
            if 'price' in col or 'interest' in col:
                df_indicators =  df_indicators.merge(indicator_func(df, col, timeframe, weighter='volume'), how='left', on='datetime')
            else:
                df_indicators =  df_indicators.merge(indicator_func(df, col, timeframe), how='left', on='datetime')

In [8]:
df_indicators.tail()

,datetime,volume_delta,cumulative_volume_delta,transaction_delta,cumulative_transaction_delta,liquidation_delta,cumulative_liquidation_delta,close_price_7_ma,close_price_7_std,close_price_7_z_score,...,close_open_interest_7_z_score,close_open_interest_31_ma,close_open_interest_31_std,close_open_interest_31_z_score,close_open_interest_127_ma,close_open_interest_127_std,close_open_interest_127_z_score,close_open_interest_367_ma,close_open_interest_367_std,close_open_interest_367_z_score
2206,1758931200,-56.719,-2.269407e+06,-4742.0,-2793313.0,3.239,215244.102,111214.584102,1905.674535,-0.859163,...,-1.193083,89019.401803,2236.601517,-1.997903,85562.818342,5271.794895,-0.191952,83312.534210,6575.382978,0.188332
2207,1759017600,2400.225,-2.267006e+06,15111.0,-2778202.0,-42.195,215201.907,111135.686685,1679.288940,0.585911,...,-0.237148,88980.157530,2281.654101,-1.266643,85558.195841,5285.377346,0.100640,83293.019346,6578.968253,0.425157
2208,1759104000,698.891,-2.266308e+06,-7791.0,-2785993.0,-42.543,215159.364,111355.393680,2024.964099,1.432967,...,1.117459,88966.944892,2297.477860,-0.044002,85560.134787,5293.400465,0.624498,83294.731654,6586.344494,0.845859
2209,1759190400,1345.801,-2.264962e+06,-4312.0,-2790305.0,-19.666,215139.698,111694.313927,2224.294882,1.031557,...,0.882130,88939.848887,2276.691066,-0.182486,85540.559013,5285.955669,0.564482,83301.030971,6588.242042,0.792830
2210,1759276800,-408.230,-2.265370e+06,-6471.0,-2796776.0,1.500,215141.198,111526.318790,2322.749070,1.258759,...,1.723048,88925.277454,2291.828434,0.365948,85528.150471,5318.806300,0.796385,83299.888442,6591.853062,0.980616


### 6 - Save as CSV file

In [9]:
# saves in CSV file    
df_indicators.to_csv("indicators.csv", index=False) 